In [1]:
pip install catboost xgboost scikit-learn==1.5.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 19.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 45.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
import gensim
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

import torch
import math
from sklearn.cluster import DBSCAN
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, RobertaModel, RobertaTokenizer

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
leetcode_questions_df = pd.read_csv('/content/drive/MyDrive/thesis/leetcode/part4 feature-engineering/leetcode_questions_df.csv')

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61834 entries, 0 to 61833
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   username                            61834 non-null  object 
 1   country                             61834 non-null  object 
 2   contest_url                         61834 non-null  object 
 3   num_of_contest                      61834 non-null  int64  
 4   is_weekly                           61834 non-null  bool   
 5   rank                                61834 non-null  int64  
 6   score                               61834 non-null  int64  
 7   question_number                     61834 non-null  int64  
 8   question_language                   61834 non-null  object 
 9   question_code                       61834 non-null  object 
 10  number_of_lines                     61834 non-null  int64  
 11  names_set                           61834

In [6]:
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_language'] == 'java']
leetcode_questions_df = leetcode_questions_df[leetcode_questions_df['question_number'] > 2]
leetcode_questions_df = leetcode_questions_df.drop_duplicates(subset=['question_code'])

leetcode_questions_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2999 entries, 20 to 61821
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   username                            2999 non-null   object 
 1   country                             2999 non-null   object 
 2   contest_url                         2999 non-null   object 
 3   num_of_contest                      2999 non-null   int64  
 4   is_weekly                           2999 non-null   bool   
 5   rank                                2999 non-null   int64  
 6   score                               2999 non-null   int64  
 7   question_number                     2999 non-null   int64  
 8   question_language                   2999 non-null   object 
 9   question_code                       2999 non-null   object 
 10  number_of_lines                     2999 non-null   int64  
 11  names_set                           2999 non-n

In [7]:
model_name = "neulab/codebert-java"
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaModel.from_pretrained(model_name)

code_snippets = leetcode_questions_df.question_code.tolist()

# Step 1: Encode the code snippets using CodeBERT
def get_embeddings(code_snippet):
    inputs = tokenizer(code_snippet, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Use the last hidden state of the [CLS] token as the embedding
    return outputs.last_hidden_state[:, 0, :].numpy()

# Get embeddings for all code snippets
embeddings = np.vstack([get_embeddings(snippet) for snippet in code_snippets])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.54k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at neulab/codebert-java and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [8]:
min_samples = 10 ** (math.floor(math.log10(len(code_snippets))) - 1)

min_samples

100

In [9]:
# Step 2: Apply DBSCAN for clustering and outlier detection
dbscan = DBSCAN(eps=0.1, min_samples=min_samples, metric='cosine', n_jobs=-1)
db_labels = dbscan.fit_predict(embeddings)

# Step 3: Identify and handle outliers
outliers = np.where(db_labels == -1)[0]

# Output some statistics
print(f'Removed {len(outliers)} outliers.')
print(f'Retained {len(db_labels) - len(outliers)} code snippets.')

Removed 115 outliers.
Retained 2884 code snippets.


In [10]:
# Remove outliers from the DataFrame
leetcode_questions_df.reset_index(drop=True, inplace=True)
leetcode_questions_df = leetcode_questions_df[~leetcode_questions_df.index.isin(outliers)]

leetcode_questions_df.country.value_counts()

,count
country,
India,1345
United States,704
Canada,173
China,172
Taiwan,83
Ukraine,59
Japan,53
Australia,50
Singapore,49


In [11]:
# Define rank thresholds
percantage = 10

high_rank_threshold = leetcode_questions_df['contest_finish_time_total_seconds'].quantile(percantage/100)
low_rank_threshold = leetcode_questions_df['contest_finish_time_total_seconds'].quantile(1 - percantage/100)


leetcode_questions_df["experienced_programmer"] = leetcode_questions_df["contest_finish_time_total_seconds"] <= low_rank_threshold


high_rank_df = leetcode_questions_df[leetcode_questions_df["contest_finish_time_total_seconds"] < high_rank_threshold]
low_rank_df = leetcode_questions_df[leetcode_questions_df["contest_finish_time_total_seconds"] > low_rank_threshold]


filtered_df = pd.concat([high_rank_df, low_rank_df])

filtered_df.reset_index(drop=True, inplace=True)

filtered_df["experienced_programmer"].value_counts()

<ipython-input-11-264e82c10e3d>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leetcode_questions_df["experienced_programmer"] = leetcode_questions_df["contest_finish_time_total_seconds"] <= low_rank_threshold


,count
experienced_programmer,
False,289
True,288


In [12]:
filtered_df.drop(['num_of_contest','is_weekly','rank','score','global_rank_percentile','num_contests_participated','question_number','country'],axis=1, inplace=True)

In [13]:
filtered_df.drop(['contest_finish_time_total_seconds','question_finish_time_total_seconds','time_spent_per_question'],axis=1, inplace=True)

In [14]:
X=filtered_df.drop('experienced_programmer',axis=1)
Y=filtered_df.experienced_programmer.astype(int)

In [15]:
X_nontext=X.select_dtypes(exclude=['object'])
X_nontext.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 577 entries, 0 to 576
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   number_of_lines              577 non-null    int64  
 1   token_count                  577 non-null    int64  
 2   variables_count              577 non-null    int64  
 3   function_count               577 non-null    int64  
 4   loop_count                   577 non-null    int64  
 5   condition_count              577 non-null    int64  
 6   single_line_comment_density  577 non-null    float64
 7   multiline_comment_density    577 non-null    float64
 8   function_density             577 non-null    float64
 9   loop_density                 577 non-null    float64
 10  condition_density            577 non-null    float64
 11  comment_tokens_density       577 non-null    float64
 12  question_code_length         577 non-null    int64  
 13  has_camel_case      

In [16]:
X_train_nontext, X_test_nontext, y_train, y_test = train_test_split(X_nontext, Y, test_size=0.2, random_state=0,stratify=Y)

In [17]:
X_train_text, X_test_text, Y_train, y_test = train_test_split(X.question_code, Y, test_size=0.2, random_state=0,stratify=Y)

In [18]:
def read_corpus(text, tokens_only=False):
    for i, line in enumerate(text):
        tokens = gensim.utils.simple_preprocess(line)
        if tokens_only:
            yield tokens
        else:
        # For training data, add tags
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

train_corpus = list(read_corpus(X_train_text))
test_corpus = list(read_corpus(X_test_text, tokens_only=True))

In [19]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=300, min_count=2)
model.build_vocab(train_corpus)
model.train(train_corpus, total_examples=model.corpus_count, epochs=55)

In [20]:
vectors = [model.infer_vector(train_corpus[doc_id].words) for doc_id in range(len(train_corpus))]
X_train_doc2vec = np.vstack(vectors)

test_vectors = [model.infer_vector(test_corpus[doc_id]) for doc_id in range(len(test_corpus))]
X_test_doc2vec = np.vstack(test_vectors)

X_train_doc2vec.shape , X_test_doc2vec.shape

((461, 300), (116, 300))

In [21]:
X_train_combined=pd.concat([pd.DataFrame(X_train_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_train_nontext.index),X_train_nontext],axis=1)

X_test_combined=pd.concat([pd.DataFrame(X_test_doc2vec, columns=['doc2vec_'+str(i) for i in range(300)], index=X_test_nontext.index),X_test_nontext],axis=1)

# Cat Boost

In [22]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',CatBoostClassifier(iterations=10))])

In [23]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

Learning rate set to 0.458724
0:	learn: 0.6284891	total: 49.2ms	remaining: 443ms
1:	learn: 0.5843551	total: 50ms	remaining: 200ms
2:	learn: 0.5596922	total: 50.7ms	remaining: 118ms
3:	learn: 0.5460567	total: 51.4ms	remaining: 77.1ms
4:	learn: 0.5275886	total: 52.1ms	remaining: 52.1ms
5:	learn: 0.5215493	total: 52.3ms	remaining: 34.9ms
6:	learn: 0.5048983	total: 53ms	remaining: 22.7ms
7:	learn: 0.4938388	total: 53.7ms	remaining: 13.4ms
8:	learn: 0.4763286	total: 54.4ms	remaining: 6.04ms
9:	learn: 0.4611587	total: 55.1ms	remaining: 0us
Learning rate set to 0.459255
0:	learn: 0.6375142	total: 1.01ms	remaining: 9.13ms
1:	learn: 0.5934587	total: 1.75ms	remaining: 6.99ms
2:	learn: 0.5756630	total: 2.46ms	remaining: 5.74ms
3:	learn: 0.5660921	total: 3.18ms	remaining: 4.78ms
4:	learn: 0.5480580	total: 3.91ms	remaining: 3.91ms
5:	learn: 0.5375091	total: 4.57ms	remaining: 3.05ms
6:	learn: 0.5231794	total: 5.25ms	remaining: 2.25ms
7:	learn: 0.5012393	total: 5.85ms	remaining: 1.46ms
8:	learn: 0.49

In [24]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6362821	total: 964us	remaining: 8.68ms
1:	learn: 0.6045991	total: 2ms	remaining: 8ms
2:	learn: 0.5777673	total: 2.74ms	remaining: 6.4ms
3:	learn: 0.5580331	total: 3.51ms	remaining: 5.26ms
4:	learn: 0.5440937	total: 4.22ms	remaining: 4.22ms
5:	learn: 0.5271247	total: 4.88ms	remaining: 3.25ms
6:	learn: 0.5160892	total: 5.64ms	remaining: 2.42ms
7:	learn: 0.5049642	total: 6.34ms	remaining: 1.58ms
8:	learn: 0.4945780	total: 7.04ms	remaining: 782us
9:	learn: 0.4898036	total: 7.31ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 1, 'classifier__learning_rate': 0.3}
Best F1 score found:  0.6994885163192845
Test set F1 score:  0.7200603318250377


# Cat with Text

In [25]:
param_grid = {
    'classifier__learning_rate': [0.01, 0.1, 0.3],
    'classifier__l2_leaf_reg': [1, 5 ,10],
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_cb = grid_search.best_estimator_
y_pred = best_cb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
0:	learn: 0.6881508	total: 17.9ms	remaining: 161ms
1:	learn: 0.6838469	total: 31.2ms	remaining: 125ms
2:	learn: 0.6790820	total: 44.2ms	remaining: 103ms
3:	learn: 0.6739608	total: 57.1ms	remaining: 85.6ms
4:	learn: 0.6690409	total: 69.9ms	remaining: 69.9ms
5:	learn: 0.6647922	total: 83.1ms	remaining: 55.4ms
6:	learn: 0.6611511	total: 96.4ms	remaining: 41.3ms
7:	learn: 0.6566073	total: 109ms	remaining: 27.4ms
8:	learn: 0.6526468	total: 122ms	remaining: 13.6ms
9:	learn: 0.6488850	total: 135ms	remaining: 0us
Best parameters found:  {'classifier__l2_leaf_reg': 1, 'classifier__learning_rate': 0.01}
Best F1 score found:  0.7163952590166425
Test set F1 score:  0.7303337082864643


# Support Vector Classifier

In [26]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',SVC(probability=True))])

In [27]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.6356007480130902


In [28]:
param_grid = {
    'classifier__C': [0.1, 1, 10, 100],
    'classifier__gamma': ['scale', 'auto'],
    'classifier__kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters found:  {'classifier__C': 100, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
Best F1 score found:  0.6965348026101377
Test set F1 score:  0.7068093963722866


# SVC with Text

In [29]:
param_grid = {
    'classifier__C': [0.01 ,0.1, 1],
    'classifier__gamma': ['scale'],
    'classifier__kernel': ['rbf', 'sigmoid']
}

grid_search = GridSearchCV(estimator=baseline_model, param_grid=param_grid,
                           cv=5, n_jobs=-1, verbose=2, scoring='f1_macro')

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best parameters found:  {'classifier__C': 1, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Best F1 score found:  0.7545918851605193
Test set F1 score:  0.701660042085574


# XGBoost

In [30]:
baseline_model = Pipeline([('scaler',StandardScaler()),
                           ('classifier',XGBClassifier())])

In [31]:
le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [32]:
scores = cross_val_score(baseline_model, X_train_nontext, y_train, cv=5)
print("baseline model score: ",np.mean(scores))

baseline model score:  0.6703833567087424


In [33]:
param_grid = {
    'n_estimators': [25, 50, 100],
    'max_depth': [5, 7, 9],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'subsample': [0.4, 0.6, 0.8],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(eval_metric='logloss')

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_nontext, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_nontext)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found:  {'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 100, 'subsample': 0.4}
Best F1 score found:  0.7217112690742258
Test set F1 score:  0.7642808760442538


In [34]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.88      0.79        58
           1       0.84      0.66      0.74        58

    accuracy                           0.77       116
   macro avg       0.78      0.77      0.76       116
weighted avg       0.78      0.77      0.76       116



# XGBoost With Text

In [35]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 7, 9],
    'learning_rate': [1e-3, 1e-2, 1e-1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_model = XGBClassifier(eval_metric='logloss', random_state=0)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train_combined, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best F1 score found: ", grid_search.best_score_)

best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_combined)
print("Test set F1 score: ", f1_score(y_test, y_pred, average='macro'))

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found:  {'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.6}
Best F1 score found:  0.7917705592551644
Test set F1 score:  0.7144775117475946


In [36]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.69      0.78      0.73        58
           1       0.75      0.66      0.70        58

    accuracy                           0.72       116
   macro avg       0.72      0.72      0.71       116
weighted avg       0.72      0.72      0.71       116

